# Cross-Check Against the National Zoning Atlas: Greenwich vs. Bridgeport

The goal of this notebook was to pull the National Zoning Atlas's (NZA) own score/classification for **Greenwich RA-2** and **Bridgeport DX1** specifically, as an independent check on our model's district-level results.

**That turned out not to be possible without an account.** Investigating the Atlas's interactive map (`https://www.zoningatlas.org/atlas`) found that individual-district details (name, specific allowances, lot-size figures) are gated behind a login. This is confirmed directly in NZA's own JavaScript source, not inferred:

```js
// anonymous users still get the clicked districts highlighted on the map (via SELECTED_ZONINGDISTRICTS),
// but the info panel itself is withheld — the remaining fields are too thin to be worth showing (issue #437)
if (! USER_IS_AUTHENTICATED) return this.collapsePanel();
```

Even the underlying API endpoint (`/atlas/zoneclick/`) returns only an internal ID, a one-letter type code, and an overlay flag for anonymous requests — no district name, no dimensional values. I did not create an NZA account to get around this (account creation on your behalf isn't something I do without you asking for it directly), so true `RA-2`-vs-`DX1` NZA data isn't in this notebook.

**What *is* publicly available, and what this notebook uses instead:** NZA publishes a free, public **Jurisdiction Snapshot** for every town — a town-wide statistical rollup (not broken out by individual district) covering what share of the town's residential land allows each housing type, minimum lot size brackets, ADU allowances, and parking mandates. This isn't a 1:1 match for our district-level model, but it's a legitimate, independently-collected second opinion on the same two towns, from the most-cited zoning dataset in the country. All figures below were read directly from NZA's own snapshot pages and visually verified against the rendered charts (not scraped blind — flattened text extraction scrambles the label-to-value order on NZA's stacked-bar charts, so every number here was confirmed against a screenshot of the actual chart, not just the raw text).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", None)

## 1. NZA Jurisdiction Snapshot data (town-wide, publicly accessible)

Source: `https://edit.zoningatlas.org/statsrollup/jurisdiction/317/` (Greenwich, CT) and `https://edit.zoningatlas.org/statsrollup/jurisdiction/311/` (Bridgeport, CT), both retrieved and visually verified against the rendered page.

In [ ]:
nza = pd.DataFrame([
    {
        "town": "Greenwich",
        "nza_jurisdiction_id": 317,
        "population": 64594,
        "land_area_acres": 30322,
        "zoning_pages_analyzed": 391,
        "total_districts": 49,
        "pct_land_zoned": 91,
        "pct_land_protected": 5,
        "pct_land_unzoned_general": 4,
        "pct_primarily_residential": 89,
        "pct_mixed_with_residential": 10,
        "pct_nonresidential": 1,
        "pct_1family_allowed": 99,
        "pct_2family_allowed": 5,   # approx remainder after ~95% prohibited
        "pct_3family_allowed": 2,   # approx remainder after ~98% prohibited
        "pct_4plusfamily_allowed": 2,  # approx remainder after ~98% prohibited
        "pct_2family_prohibited": 95,
        "pct_3family_prohibited": 98,
        "pct_4plusfamily_prohibited": 98,
        "pct_1family_minlot_none": 10,
        "pct_1family_minlot_80k_or_more_sqft": 60,
        "pct_adu_allowed_any_form": 87,  # NZA 'Allowed' bucket (not combined w/ restricted/hearing variants)
        "pct_adu_prohibited": 12,
        "pct_residential_land_with_parking_mandate": 92,
    },
    {
        "town": "Bridgeport",
        "nza_jurisdiction_id": 311,
        "population": 151599,
        "land_area_acres": 10246,
        "zoning_pages_analyzed": 301,
        "total_districts": 25,
        "pct_land_zoned": 72,
        "pct_land_protected": 6,
        "pct_land_unzoned_general": 22,
        "pct_primarily_residential": 56,
        "pct_mixed_with_residential": 23,
        "pct_nonresidential": 20,
        "pct_1family_allowed": 83,
        "pct_2family_allowed": 50,
        "pct_3family_allowed": 61,
        "pct_4plusfamily_allowed": 61,
        "pct_2family_prohibited": 32,
        "pct_3family_prohibited": 39,
        "pct_4plusfamily_prohibited": 38,
        "pct_1family_minlot_none": 81,
        "pct_1family_minlot_80k_or_more_sqft": 0,
        "pct_adu_allowed_any_form": 23,
        "pct_adu_prohibited": 54,
        "pct_residential_land_with_parking_mandate": 0,
    },
])
nza

## 2. A simple NZA-based restrictiveness proxy, for comparison against our model

This is **not** the same measurement as our district-level Restrictiveness Score — it's a town-wide average, not a single-district score, and NZA's own public dashboard doesn't publish a single composite number at all (that's a deliberate design choice on their part; they present the underlying shares and let the reader draw conclusions). To make a rough comparison possible, this builds an analogous composite here: the average "prohibited" share across 2/3/4+-family housing types (the closest NZA equivalent to our model's Use Exclusion index), plus the large-lot share and the parking-mandate prevalence, each weighted the same way our own Index B / Index A / Index E are weighted, with Index C/D omitted since NZA's public snapshot doesn't break out setbacks/height/coverage at all.

In [ ]:
def nza_restrictiveness_proxy(row):
    use_exclusion = np.mean([
        row["pct_2family_prohibited"],
        row["pct_3family_prohibited"],
        row["pct_4plusfamily_prohibited"],
    ])
    density_restriction = row["pct_1family_minlot_80k_or_more_sqft"]
    procedural_friction = row["pct_residential_land_with_parking_mandate"]

    # same relative weights as our Index B (0.30) and Index A (0.25) and Index E (0.15) in the
    # main model, renormalized since C and D have no NZA public equivalent (0.30+0.25+0.15=0.70 -> /0.70)
    return (0.30 * use_exclusion + 0.25 * density_restriction + 0.15 * procedural_friction) / 0.70


nza["nza_restrictiveness_proxy"] = nza.apply(nza_restrictiveness_proxy, axis=1)
nza[["town", "nza_restrictiveness_proxy"]]

## 3. Side-by-side comparison chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Panel 1: our model's two example districts (from the main notebook's last run) ---
our_model = pd.DataFrame({
    "label": ["Greenwich RA-2\n(our model)", "Bridgeport DX1\n(our model)"],
    "score": [67.8, 19.9],
})
axes[0].bar(our_model["label"], our_model["score"], color=["#c0392b", "#2980b9"])
axes[0].set_ylim(0, 100)
axes[0].set_ylabel("Restrictiveness (0=permissive, 100=restrictive)")
axes[0].set_title("Our district-level model\n(single district, full dataset)")
axes[0].axhline(50, color="gray", linestyle="--", linewidth=0.8)

# --- Panel 2: NZA town-wide proxy ---
axes[1].bar(
    nza["town"] + "\n(NZA town-wide)",
    nza["nza_restrictiveness_proxy"],
    color=["#c0392b", "#2980b9"],
)
axes[1].set_ylim(0, 100)
axes[1].set_title("NZA public snapshot data\n(town-wide average, not district-level)")
axes[1].axhline(50, color="gray", linestyle="--", linewidth=0.8)

plt.suptitle("Greenwich vs. Bridgeport: two independent measurements point the same direction", y=1.03)
plt.tight_layout()
plt.savefig("../data/collected_data/nza_comparison_chart.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. What this does and doesn't prove

**Directionally, the two approaches agree completely:** both our district-level model and NZA's independent town-wide statistics show Greenwich dramatically more restrictive than Bridgeport, by a wide margin, on every comparable dimension:

| Metric | Greenwich | Bridgeport |
|---|---|---|
| 2-family prohibited | 95% of residential land | 32% of residential land |
| 3-family prohibited | 98% of residential land | 39% of residential land |
| 4+-family prohibited | 98% of residential land | 38% of residential land |
| 1-family lots requiring 80,000+ sqft | 60% of 1-family land | 0% |
| Residential land with a parking mandate | 92% | 0% |
| ADU prohibited | 12% | 54% (note: Bridgeport is *more* ADU-restrictive than Greenwich, the one metric where the ranking flips) |

**What this is NOT:** a validation of our exact score (67.8 vs. 19.9) against an NZA-reported score for the same two districts — that comparison isn't possible without an NZA account, since district-level detail is login-gated. It's also town-wide vs. district-specific: Greenwich's 91% zoned / 89% primarily-residential town-wide figures are an average across 49 districts including some less restrictive than RA-2, so a literal number-for-number match was never going to be possible even with access.

**The one genuine surprise:** ADU policy reverses the overall pattern — Bridgeport prohibits ADUs on more of its land (54%) than Greenwich does (12%), even though Bridgeport is far more permissive on everything else. That's a real, useful finding for the ROAD Act angle: it shows exclusion isn't always monotonic with "restrictive town" vs. "permissive town" — it's worth checking variable-by-variable rather than assuming a uniformly restrictive or permissive profile.

**If district-level NZA data is wanted later:** that requires either (a) creating a free NZA account yourself and re-running the `/atlas/zoneclick/` lookup while authenticated, or (b) requesting the underlying dataset directly from NZA/Land Use Atlas, Inc. for research use. I didn't do either here since account creation wasn't something you asked for.